## 1. Data Collection

In [7]:
import pandas as pd

train_df = pd.read_csv('../data/fraudTrain.csv')
test_df = pd.read_csv('../data/fraudTest.csv')


In [8]:
train_df.head(5)

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


### About the Dataset (23 columns)

1. `Unnamed: 0` — copy of index
2. `trans_date_trans_time` — transaction date and time
3. `cc_num` — card number
4. `merchant` — merchant name
5. `category` — transaction category
6. `amt` — transaction amount ($)
7. `first` — customer name
8. `last` — customer last name
9. `gender` — customer gender
10. `street` — customer street address
11. `city` — customer city
12. `state` — customer state
13. `zip` — customer zip code
14. `lat` — customer latitude
15. `long` — customer longitude
16. `city_pop` — city population
17. `job` — customer occupation
18. `dob` — customer date of birth
19. `trans_num` — transaction ID
20. `unix_time` — transaction time (unix format)
21. `merch_lat` — merchant latitude
22. `merch_long` — merchant longitude
23. `is_fraud` — target variable (0=normal, 1=fraud)

In [9]:
train_df.shape

(1296675, 23)

In [10]:
train_df.columns.tolist()

['Unnamed: 0',
 'trans_date_trans_time',
 'cc_num',
 'merchant',
 'category',
 'amt',
 'first',
 'last',
 'gender',
 'street',
 'city',
 'state',
 'zip',
 'lat',
 'long',
 'city_pop',
 'job',
 'dob',
 'trans_num',
 'unix_time',
 'merch_lat',
 'merch_long',
 'is_fraud']

In [11]:
train_df = pd.read_csv('../data/fraudTrain.csv')
test_df = pd.read_csv('../data/fraudTest.csv')

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df['is_fraud'].value_counts()

Train shape: (1296675, 23)
Test shape: (555719, 23)


is_fraud
0    1289169
1       7506
Name: count, dtype: int64

## 2. Exploratory Data Analysis / EDA 

In [12]:
train_df.groupby('is_fraud')['amt'].mean()

is_fraud
0     67.667110
1    531.320092
Name: amt, dtype: float64

This is a very significant signal! Fraudulent transactions are approximately 8 times higher in value than normal transactions. This suggests that fraudsters tend to try to make high-value purchases quickly once they gain access to a card, rather than making small "test" transactions (they want to maximize their "profit" before the card is blocked). Therefore, the amt column appears to be one of the model's strongest features.

Let's look at which categories fraudulent transactions are concentrated in for example, is there more fraud in online shopping or in grocery shopping?

This will show the percentage of transactions in each category that are fraudulent, sorted from highest to lowest.

In [13]:
print(train_df.groupby('category')['is_fraud'].mean().sort_values(ascending=False))

category
shopping_net      0.017561
misc_net          0.014458
grocery_pos       0.014098
shopping_pos      0.007225
gas_transport     0.004694
misc_pos          0.003139
grocery_net       0.002948
travel            0.002864
entertainment     0.002478
personal_care     0.002424
kids_pets         0.002114
food_dining       0.001651
home              0.001608
health_fitness    0.001549
Name: is_fraud, dtype: float64


Categories with the highest fraud rates:

shopping_net (online shopping): 1.76%
misc_net (other online): 1.45%
grocery_pos (grocery, physical POS): 1.41%
The lowest:

health_fitness: 0.15%
home: 0.16%
food_dining: 0.17%

Let's look at another pattern: Time-based
Let's also look at it by time, because fraud is generally concentrated during nighttime hours. First, we need to determine the time:

In [14]:
train_df['trans_date_trans_time'] = pd.to_datetime(train_df['trans_date_trans_time'])
train_df['hour'] = train_df['trans_date_trans_time'].dt.hour

test_df['trans_date_trans_time'] = pd.to_datetime(test_df['trans_date_trans_time'])
test_df['hour'] = test_df['trans_date_trans_time'].dt.hour


print(train_df.groupby('hour')['is_fraud'].mean().sort_values(ascending=False))

hour
22    0.028829
23    0.028374
1     0.015349
0     0.014940
2     0.014652
3     0.014239
5     0.001423
7     0.001327
14    0.001325
19    0.001236
18    0.001226
13    0.001225
15    0.001208
17    0.001192
16    0.001156
8     0.001153
21    0.001129
9     0.001114
4     0.001099
12    0.001027
11    0.000998
20    0.000952
10    0.000946
6     0.000946
Name: is_fraud, dtype: float64


This code is actually both an exploratory step and a small feature engineering step.

So what does this mean? This perfectly reflects real-world fraud behavior: fraudsters typically conduct transactions between midnight and early morning (10 PM - 3 AM). The logic is simple! These are the hours when the cardholder is asleep and unable to notice the transaction and intervene immediately. By the time the bank/user wakes up and notices, hours have passed.

## 3. Preprocessing / Data Manipulation


Let's start by dropping unnessasary columns

columns_to_drop = ['Unnamed: 0', 'first', 'last', 'street', 'cc_num', 'trans_num']

train_df = train_df.drop(columns=columns_to_drop)
test_df = test_df.drop(columns=columns_to_drop)

train_df.shape

In [15]:
columns_to_drop = ['Unnamed: 0', 'first', 'last', 'street', 'cc_num', 'trans_num']

train_df = train_df.drop(columns=columns_to_drop, errors='ignore')
test_df = test_df.drop(columns=columns_to_drop, errors='ignore')

train_df.shape

(1296675, 18)

In [16]:
test_df.shape

(555719, 18)

### Dropped Columns

The following columns were removed because they don't carry useful signal for fraud detection:

- `Unnamed: 0` — duplicate of the row index, no information
- `first` — customer first name, a random identifier unrelated to fraud
- `last` — customer last name, same reason
- `street` — customer's street address
- `cc_num` — card number, an identifier rather than a behavioral feature
- `trans_num` — unique per transaction, carries no repeatable pattern

**Result:** 23 columns → 17 columns (after also adding `hour`, currently at 18)

Let's calculate age from dob. In a new code cell:


In [17]:
train_df['dob'] = pd.to_datetime(train_df['dob'])
test_df['dob'] = pd.to_datetime(test_df['dob'])

train_df['age'] = (train_df['trans_date_trans_time'] - train_df['dob']).dt.days // 365
test_df['age'] = (test_df['trans_date_trans_time'] - test_df['dob']).dt.days // 365

print(train_df[['dob', 'trans_date_trans_time', 'age']].head())

         dob trans_date_trans_time  age
0 1988-03-09   2019-01-01 00:00:18   30
1 1978-06-21   2019-01-01 00:00:44   40
2 1962-01-19   2019-01-01 00:00:51   56
3 1967-01-12   2019-01-01 00:01:16   52
4 1986-03-28   2019-01-01 00:03:06   32


- Converts dob to a proper datetime type (same as we did for trans_date_trans_time)
- Calculates age by subtracting birth date from transaction date, getting the difference in days, then dividing by 365 to approximate years
- // is integer division — it rounds down, so someone who's "34.8 years old" becomes 34, matching how age normally works (you don't turn 35 until your birthday actually passes)
- Applied to both train and test, as always

In [18]:
print(train_df['age'].min())
print(train_df['age'].max())

13
95


Both are believable, though 13 is worth a small pause.

- Max = 95: Perfectly plausible. Someone born in the 1920s-30s still transacting in 2019-2020. No issue there.
- Min = 13: This is technically unusual. Most credit card issuers require cardholders to be 18+ (in the US, minors can be authorized users on a parent's account, but the account itself, and cc_num, would typically belong to an adult). A few possibilities:

Legitimate but unusual: some card products do allow authorized minor users, especially teens carrying a card linked to a parent's account
- Data quality artifact: since this is a simulated dataset (Sparkov generated, not real transactions), the generator may not have enforced a minimum age when creating synthetic customer profiles
- Not actually a data error, just an edge case worth noting but not necessarily something to fix

Let's see how common this is before deciding anything:

In [19]:
print(train_df[train_df['age'] < 18]['age'].value_counts().sort_index())

age
13      69
14    3855
15    6108
16    3125
17    2351
Name: count, dtype: int64


**Data quality note:** Age ranges from 13 to 95. Ages under 18 account for about 1.2% of transactions (69 to 6,108 rows per age from 13 to 17). These are kept as is since they likely represent authorized users on a parent's account, and the dates parse correctly with no signs of corruption.

In [20]:
print(train_df.groupby('is_fraud')['age'].agg(['mean', 'std', 'min', 'max']))

               mean        std  min  max
is_fraud                                
0         45.511960  17.398818   13   95
1         48.321609  18.864543   14   93


Age is a real but weak signal here, consistent with what we said earlier. It's not going to be a headline feature like amt or hour, but it's not noise either, there is a small, consistent shift toward older ages in the fraud group.

In [21]:
train_df.head()

,trans_date_trans_time,merchant,category,amt,gender,city,state,zip,lat,long,city_pop,job,dob,unix_time,merch_lat,merch_long,is_fraud,hour,age
0,2019-01-01 00:00:18,"fraud_Rippin, Kub and Mann",misc_net,4.97,F,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,1325376018,36.011293,-82.048315,0,0,30
1,2019-01-01 00:00:44,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,F,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1325376044,49.159047,-118.186462,0,0,40
2,2019-01-01 00:00:51,fraud_Lind-Buckridge,entertainment,220.11,M,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,1325376051,43.150704,-112.154481,0,0,56
3,2019-01-01 00:01:16,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,M,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,1967-01-12,1325376076,47.034331,-112.561071,0,0,52
4,2019-01-01 00:03:06,fraud_Keeling-Crist,misc_pos,41.96,M,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,1325376186,38.674999,-78.632459,0,0,32


Two common encoding approaches
1. Label Encoding: assigns each category a number (e.g. M=0, F=1). Simple, but implies an order that doesn't really exist (is "category 5" greater than "category 2"? not meaningfully). Works fine for tree-based models like XGBoost though, since trees split on thresholds rather than assuming linear order.
2. One-Hot Encoding: creates a separate 0/1 column for each category (e.g. category_travel, category_grocery_pos, etc). No false ordering implied, but can explode your column count if a feature has many unique values (like merchant or job, which could have hundreds).

## What we can do for this dataset

gender: simple label encoding (M/F, only 2 values, no ordering issue)
category: one-hot encoding (manageable number of categories, ~14)
merchant, city, job: these have too many unique values for one-hot (would create hundreds of columns). Simplest approach for a first model: drop them for now, and treat this as a limitation we can revisit later with more advanced encoding techniques

Let's check how many unique values merchant, city, and job actually have, to confirm they're too many:

In [22]:
print(train_df['merchant'].nunique())
print(train_df['city'].nunique())
print(train_df['job'].nunique())
print(train_df['state'].nunique())

693
894
494
51


Confirms it. merchant (693), city (894), and job (494) all have far too many unique values to one-hot encode without exploding the dataset into thousands of columns, that would slow training dramatically and likely hurt the model more than help it. state at 51 is borderline (matches the 50 US states plus DC), still a lot but more manageable if we wanted it later.
Plan for this step
Drop (too many unique values, not worth the complexity for a first model):

merchant
city
job
state (borderline, but let's keep things simple for now)

Encode:

gender: label encode (M/F to 0/1)
category: one-hot encode (~14 categories, manageable)

Let's do the drop first, in a new cell:

In [23]:
columns_to_drop_2 = ['merchant', 'city', 'job', 'state']

train_df = train_df.drop(columns=columns_to_drop_2, errors='ignore')
test_df = test_df.drop(columns=columns_to_drop_2, errors='ignore')

train_df.shape

(1296675, 15)

In [24]:
train_df['gender'] = train_df['gender'].map({'M': 0, 'F': 1})
test_df['gender'] = test_df['gender'].map({'M': 0, 'F': 1})

train_df['gender'].value_counts()

gender
1    709863
0    586812
Name: count, dtype: int64

This replaces every "M" with 0 and every "F" with 1, directly. Simple mapping, no library needed for just two values.

In [25]:
train_df = pd.get_dummies(train_df, columns=['category'], drop_first=True)
test_df = pd.get_dummies(test_df, columns=['category'], drop_first=True)

train_df.shape

(1296675, 27)

What pd.get_dummies does: it takes the category column and creates a new 0/1 column for each unique category value (e.g. category_grocery_pos, category_travel, etc). 
- A row gets a 1 in the column matching its actual category, and 0 in all others.
Why drop_first=True: if we have 14 categories, we only need 13 new columns, not 14.
-  The 14th category is implied when all other 13 columns are 0. This avoids redundant information (a concept called the "dummy variable trap"), which some models are sensitive to.

In [26]:
train_df.head(5)

,trans_date_trans_time,amt,gender,zip,lat,long,city_pop,dob,unix_time,merch_lat,...,category_grocery_pos,category_health_fitness,category_home,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel
0,2019-01-01 00:00:18,4.97,1,28654,36.0788,-81.1781,3495,1988-03-09,1325376018,36.011293,...,False,False,False,False,True,False,False,False,False,False
1,2019-01-01 00:00:44,107.23,1,99160,48.8878,-118.2105,149,1978-06-21,1325376044,49.159047,...,True,False,False,False,False,False,False,False,False,False
2,2019-01-01 00:00:51,220.11,0,83252,42.1808,-112.2620,4154,1962-01-19,1325376051,43.150704,...,False,False,False,False,False,False,False,False,False,False
3,2019-01-01 00:01:16,45.00,0,59632,46.2306,-112.1138,1939,1967-01-12,1325376076,47.034331,...,False,False,False,False,False,False,False,False,False,False
4,2019-01-01 00:03:06,41.96,0,24433,38.4207,-79.4629,99,1986-03-28,1325376186,38.674999,...,False,False,False,False,False,True,False,False,False,False


In [27]:
train_df.columns.tolist()

['trans_date_trans_time',
 'amt',
 'gender',
 'zip',
 'lat',
 'long',
 'city_pop',
 'dob',
 'unix_time',
 'merch_lat',
 'merch_long',
 'is_fraud',
 'hour',
 'age',
 'category_food_dining',
 'category_gas_transport',
 'category_grocery_net',
 'category_grocery_pos',
 'category_health_fitness',
 'category_home',
 'category_kids_pets',
 'category_misc_net',
 'category_misc_pos',
 'category_personal_care',
 'category_shopping_net',
 'category_shopping_pos',
 'category_travel']

What's left before we can train
Looking at this list, there are still two columns that aren't numeric and need handling:

- trans_date_trans_time: still a full datetime, XGBoost can't use this directly. We already extracted hour from it, so at this point it's done its job and can be dropped.
- dob: same story, already used to calculate age, no longer needed in its raw form.

In [28]:
train_df = train_df.drop(columns=['trans_date_trans_time', 'dob'], errors='ignore')
test_df = test_df.drop(columns=['trans_date_trans_time', 'dob'], errors='ignore')

train_df.shape


(1296675, 25)

In [29]:
test_df.shape

(555719, 25)

In [30]:
train_df.shape

(1296675, 25)

Now, both datasets has 25 columns ! 

In [31]:
train_df.head(5)

,amt,gender,zip,lat,long,city_pop,unix_time,merch_lat,merch_long,is_fraud,...,category_grocery_pos,category_health_fitness,category_home,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel
0,4.97,1,28654,36.0788,-81.1781,3495,1325376018,36.011293,-82.048315,0,...,False,False,False,False,True,False,False,False,False,False
1,107.23,1,99160,48.8878,-118.2105,149,1325376044,49.159047,-118.186462,0,...,True,False,False,False,False,False,False,False,False,False
2,220.11,0,83252,42.1808,-112.2620,4154,1325376051,43.150704,-112.154481,0,...,False,False,False,False,False,False,False,False,False,False
3,45.00,0,59632,46.2306,-112.1138,1939,1325376076,47.034331,-112.561071,0,...,False,False,False,False,False,False,False,False,False,False
4,41.96,0,24433,38.4207,-79.4629,99,1325376186,38.674999,-78.632459,0,...,False,False,False,False,False,True,False,False,False,False


This one's trickier. It's stored as a number, but it's not really a quantity, it's a location code. The model might still find some use in it (nearby zip codes are often geographically close, so there could be a loose pattern), but we already have lat and long which capture location more precisely and meaningfully. Keeping zip alongside lat/long is somewhat redundant and adds noise without much added value. Let's drop it too for a cleaner first model.


Now let's handle the two remaining questionable columns: unix_time and zip.
- unix_time:

This is the same timestamp as trans_date_trans_time, just represented as a raw number (seconds since Jan 1, 1970) instead of a readable date. Since we already extracted the useful part (hour) from the datetime version, unix_time is redundant, it doesn't add new information, just a different encoding of something we've already captured. Safe to drop.
- zip:

This one's trickier. It's stored as a number, but it's not really a quantity, it's a location code. The model might still find some use in it (nearby zip codes are often geographically close, so there could be a loose pattern), but we already have lat and long which capture location more precisely and meaningfully. Keeping zip alongside lat/long is somewhat redundant and adds noise without much added value. Let's drop it too for a cleaner first model.

In [32]:
train_df = train_df.drop(columns=['unix_time', 'zip'], errors='ignore')
test_df = test_df.drop(columns=['unix_time', 'zip'], errors='ignore')

train_df.shape

(1296675, 23)

we're expecting 23 columns now. Once confirmed, we'll do one final check (make sure there are no missing/null values anywhere), then we're ready to actually train the first model.

Now, before decomposing into the model, let's do one last check: are there any missing (null/NaN) values?

In [33]:
train_df.isnull().sum()
test_df.isnull().sum()

amt                        0
gender                     0
lat                        0
long                       0
city_pop                   0
merch_lat                  0
merch_long                 0
is_fraud                   0
hour                       0
age                        0
category_food_dining       0
category_gas_transport     0
category_grocery_net       0
category_grocery_pos       0
category_health_fitness    0
category_home              0
category_kids_pets         0
category_misc_net          0
category_misc_pos          0
category_personal_care     0
category_shopping_net      0
category_shopping_pos      0
category_travel            0
dtype: int64

Clean, no missing values, let's officially close the preprocessing phase and move on to the modeling phase.

## 4. Model Training

In [34]:
X_train = train_df.drop('is_fraud', axis=1)
y_train = train_df['is_fraud']

X_test = test_df.drop('is_fraud', axis=1)
y_test = test_df['is_fraud']

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(1296675, 22) (1296675,)
(555719, 22) (555719,)


What does this code do?

- X_train = all columns except is_fraud (inputs the model will use for learning)
- y_train = only the is_fraud column (the target the model will try to predict)
We do the same for the test data.

Let's train the XGBoost model.

In [35]:
from xgboost import XGBClassifier

model = XGBClassifier(random_state=42)
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

What does this code do?

- XGBClassifier = XGBoost's classification version, as we discussed earlier, it works by building successive decision trees, each correcting the error of the previous one.
- random_state=42 = fixes the parts of the model that use internal randomness (e.g., data sampling), so you get the same result every time you run it (standard practice for repeatability, the number 42 has no special meaning, just a popular tradition).
- .fit(X_train, y_train) = the command "learn to predict this y from these X's", the actual training takes place here.

We will measure how well the model works.

Make a prediction and evaluate.

In [37]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    553574
           1       0.91      0.76      0.83      2145

    accuracy                           1.00    555719
   macro avg       0.95      0.88      0.91    555719
weighted avg       1.00      1.00      1.00    555719



What does this code do?

- model.predict(X_test) = The model makes predictions on test data it has never seen before (0 or 1 for each row)
- classification_report = A summary table showing metrics such as precision, recall, and f1-score, both separately for each class (0 and 1) and overall.

We need to focus on the Class 1 (fraud) row.
- Precision: 0.91 → When the model says "this is fraud," there's a 91% chance it's actually fraud. So the model doesn't give many false alarms (only 9% false positives).
- Recall: 0.76 → The model was able to detect 76% of the actual fraud transactions. The remaining 24% (approximately 515 transactions, 24% of 2145) were missed.
- F1-score: 0.83 → A balanced average of precision and recall summarizes the overall performance at a glance.

F1 is somewhere in between (not exactly in the middle, but close), which tells us "the model is good but not perfect, the precision side is a bit stronger." Because it's a single number, it's useful for quickly seeing which one performs more balanced overall when comparing different models (for example, if you train a second model with different settings in the future).

In [38]:
print(confusion_matrix(y_test, y_pred))

[[553410    164]
 [   517   1628]]


-                   Normal         Fraud

-Gerçek: Normal    553,410           164

-Gerçek: Fraud         517         1,628

What does this mean?

- True Negative = 553,410 → The model correctly identified the vast majority of truly normal transactions as "normal." This is expected, as the majority of the dataset consists of normal transactions.

- False Positive = 164 → The model mistakenly flagged 164 normal transactions as "fraud." In real life, this would mean 164 innocent customers having their cards unnecessarily blocked or receiving "suspicious transaction" warnings. It's a disturbing but relatively small number (only 0.03% of 553,574 normal transactions).

- False Negative = 517 → This is the most critical number. The model missed 517 genuine fraud transactions, labeling them as "normal.These are actual fraud cases where the bank/customer loses money, and the system fails to detect them. 

-True Positive = 1,628 → The model correctly identified 1,628 out of a total of 2,145 fraud transactions. This is the raw equivalent of recall (0.76) in the precision/recall calculation: 1628 / (1628+517) = 0.76.


### The model currently catches 1628 real frauds compared to 164 false alarms, but misses 517. This is a fairly reasonable initial balance, catching the majority of frauds (approximately 3 out of 4) with very few false alarms (without significantly disrupting customer experience).

### Where is the opportunity for improvement?
- The 517 missed frauds are the main area that can be worked on in the future. Generally, the following are tried:

Adjusting model parameters (hyperparameter tuning)
Changing the decision threshold (currently the model says "fraud" if it gives a probability above 50%, lowering this threshold can catch more frauds but at the cost of more false alarms)
Adding more/different features

## 5. Model Improvement

### 5.1 Grid Search with Class Imbalance Handling


In [39]:
from sklearn.model_selection import GridSearchCV

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

param_grid = {
    'max_depth': [4, 6],
    'n_estimators': [100, 200],
}

base_model = XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight)

grid_search = GridSearchCV(base_model, param_grid, scoring='f1', cv=3)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)

Best parameters: {'max_depth': 6, 'n_estimators': 200}


The best combination: max_depth=6, n_estimators=200. Of the 4 combinations we tried (max_depth: 4/6, n_estimators: 100/200), the one with the deepest tree and the most trees won, meaning it gives a better F1 score when the model is a bit more complex/larger.
 - We need to retrain a model with these best parameters and evaluate it to compare it to the first model.

### 5.2 Retrain with Best Parameters

In [40]:
best_model = XGBClassifier(
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    max_depth=6,
    n_estimators=200
)

best_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

### 5.3 Evaluate Improved Model

In [41]:
y_pred_improved = best_model.predict(X_test)

print(classification_report(y_test, y_pred_improved))
print(confusion_matrix(y_test, y_pred_improved))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    553574
           1       0.46      0.88      0.60      2145

    accuracy                           1.00    555719
   macro avg       0.73      0.94      0.80    555719
weighted avg       1.00      1.00      1.00    555719

[[551341   2233]
 [   261   1884]]


What happened?
scale_pos_weight did exactly what it promised: we made the model say "missing fraud is much worse than giving a false alarm," and the model complied.
Good news: Recall increased from 0.76 to 0.88, meaning the number of missed frauds dropped from 517 to 261, almost halving. The number of detected frauds increased from 1628 to 1884.
Bad news: Precision plummeted from 0.91 to 0.46. The number of false alarms jumped from 164 to 2233, meaning that now only 1 out of every 2 transactions the model calls "fraud" is actually fraud, the other is an innocent transaction.
Why did the F1 score drop (0.83 → 0.60)?
As you may recall, F1 is a metric that tries to keep both precision and recall balanced. Here, the precision dropped so sharply (0.91→0.46) that the gain in recall (0.76→0.88) couldn't compensate, and F1 as a whole deteriorated. Is this a "failure"?
No, it's actually a very valuable learning moment. It shows us that scale_pos_weight was set too aggressively; we directly used the class imbalance ratio (around 172, because that's the normal/fraud ratio), but this was a very harsh correction. In real life, is it a reasonable price to pay for inconveniencing 2233 innocent customers (their cards are blocked, they receive warnings) to catch 256 extra fraudsters? It depends on the business context, but in our case, the F1 score clearly worsened.